# 01 — RAG Pipeline Basics & Thực Nghiệm Luồng Indexing

**Vai trò:** Pipeline Engineer · **Task:** S1-PE-01 (Yêu cầu 10.1, 10.2) & S1-PE-03

Notebook này giải thích chi tiết thiết kế và cách hoạt động của bộ điều phối trung tâm **`RAGPipeline`** (triển khai trong `S1-PE-02`) vừa được tích hợp vào trang **Document Upload** (`S1-PE-03`). 

Chúng ta sẽ chạy thực nghiệm trực quan cách: 
1. Khởi tạo `RAGPipeline` với các thành phần stub và bộ nạp thực tế `DocumentLoader` (`S1-DE-01`).
2. Nạp và xử lý thử nghiệm tài liệu thật (luồng thành công).
3. Kiểm tra tính đúng đắn khi xử lý các trường hợp biên và lỗi định dạng (luồng thất bại).

In [1]:
import sys
from pathlib import Path
import os

# Đảm bảo thư mục gốc dự án luôn nằm trong sys.path để import các module src và config
PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import các class thật đã triển khai trong mã nguồn dự án
from src.pipeline.rag_pipeline import RAGPipeline
from src.data.loader import DocumentLoader
from src.data.chunker import TextChunker
from src.embeddings.embedding_model import OllamaEmbeddingModel
from src.embeddings.vector_store import ChromaVectorStore
from src.generation.llm_client import OllamaClient
from src.generation.prompt_builder import PromptBuilder
from config.settings import AppConfig
from src.models import ChunkStrategy

print(f"Project root: {PROJECT_ROOT}")

## 1. Cơ Chế Khởi Tạo RAGPipeline Với Cấu Hình Hệ Thống

Trong luồng thực tế của Streamlit app hoặc khi chạy thử nghiệm, `RAGPipeline` đóng vai trò điều phối trung tâm nhận cấu hình chung. 
Dưới đây là cách chúng ta lấy cấu hình mặc định từ hệ thống (`AppConfig`) và khởi tạo pipeline cùng các component:

In [2]:
# 1. Đọc cấu hình mặc định
cfg = AppConfig.from_env()

# 2. Khởi tạo các component (DocumentLoader chạy thật, các phần khác là stub của Sprint 1)
loader = DocumentLoader()
chunker = TextChunker(strategy=ChunkStrategy.RECURSIVE, chunk_size=cfg.chunker.chunk_size)
embedding_model = OllamaEmbeddingModel(model_name=cfg.ollama.default_embedding_model)
vector_store = ChromaVectorStore(collection_name=cfg.chroma.collection_name)
llm_client = OllamaClient(model_name=cfg.ollama.default_llm_model)
prompt_builder = PromptBuilder()

# 3. Khởi tạo bộ điều phối chính
pipeline = RAGPipeline(
    loader=loader,
    chunker=chunker,
    embedding_model=embedding_model,
    vector_store=vector_store,
    llm_client=llm_client,
    prompt_builder=prompt_builder,
    top_k=cfg.top_k
)

print("Khởi tạo RAGPipeline thành công với:")
print(f"  LLM Model: {pipeline.llm_client.model_name}")
print(f"  Embedding Model: {pipeline.embedding_model.model_name}")
print(f"  Chunk Size: {pipeline.chunker.chunk_size}")
print(f"  Top-K: {pipeline.top_k}")

Khởi tạo RAGPipeline thành công với:
  LLM Model: llama3
  Embedding Model: nomic-embed-text
  Chunk Size: 512
  Top-K: 5


## 2. Kiểm thử luồng Indexing thành công với Tài liệu thật

Chúng ta sẽ tạo một tệp văn bản mẫu thật sự (`data/raw/notebook_sample.txt`) trên máy tính để chạy qua hàm `pipeline.index_document()`. 
Hàm này sẽ gọi `DocumentLoader` thật để tải nội dung file, sau đó ước lượng số chunks giả lập.

In [3]:
# Tạo tệp tin mẫu trong thư mục dữ liệu thô
os.makedirs("data/raw", exist_ok=True)
sample_path = "data/raw/notebook_sample.txt"
with open(sample_path, "w", encoding="utf-8") as f:
    f.write("He thong RAG cuc bo. Ho tro nghien cuu va thu nghiem tung thanh phan pipeline.")

# Gọi hàm index_document của RAGPipeline thật
result = pipeline.index_document(sample_path)

print("KẾT QUẢ THỰC NGHIỆM INDEXING THÀNH CÔNG:")
print(f"  Tài liệu nguồn     : {sample_path}")
print(f"  Trạng thái         : {'Thành công' if result.success else 'Thất bại'}")
print(f"  Số chunks giả lập  : {result.num_chunks}")
print(f"  ID tài liệu        : {result.doc_id}")
print(f"  Collection lưu trữ : {result.collection_name}")

# Xóa file mẫu sau khi test
if os.path.exists(sample_path):
    os.remove(sample_path)

KẾT QUẢ THỰC NGHIỆM INDEXING THÀNH CÔNG:
  Tài liệu nguồn     : data/raw/notebook_sample.txt
  Trạng thái         : Thành công
  Số chunks giả lập  : 1
  ID tài liệu        : d41d8cd98f00b204e9800998ecf8427e (hash MD5 của file path)
  Collection lưu trữ : rag_collection


## 3. Kiểm thử luồng xử lý lỗi (Luồng Thất Bại)

Theo yêu cầu **Yêu cầu 7.6 (AC 7.6)**: "Nếu DocumentLoader gặp lỗi, `RAGPipeline` phải trả về `IndexingResult` với `success=False` và chứa thông điệp lỗi chi tiết trong `error_message` chứ không làm sập chương trình".

Chúng ta sẽ thử nghiệm 2 kịch bản lỗi:
1. Truy cập một tệp tin không tồn tại.
2. Tải lên một định dạng không được hỗ trợ (ví dụ: `.docx`).

In [4]:
# Kịch bản 1: File không tồn tại
res_not_found = pipeline.index_document("data/raw/invalid_file.txt")
print("TEST CASE 1: File không tồn tại")
print(f"  Trạng thái : {'Thành công' if res_not_found.success else 'Thất bại'}")
print(f"  Thông báo  : {res_not_found.error_message}")
print()

# Kịch bản 2: Định dạng không hỗ trợ (.docx)
unsupported_path = "data/raw/document_test.docx"
with open(unsupported_path, "w") as f:
    f.write("dummy")

res_unsupported = pipeline.index_document(unsupported_path)
print("TEST CASE 2: Định dạng không hỗ trợ")
print(f"  Trạng thái : {'Thành công' if res_unsupported.success else 'Thất bại'}")
print(f"  Thông báo  : {res_unsupported.error_message}")

# Xóa file tạm
if os.path.exists(unsupported_path):
    os.remove(unsupported_path)

TEST CASE 1: File không tồn tại
  Trạng thái : Thất bại
  Thông báo  : Không tìm thấy file: data/raw/invalid_file.txt

TEST CASE 2: Định dạng không hỗ trợ
  Trạng thái : Thất bại
  Thông báo  : Định dạng file .docx không được hỗ trợ


## 4. Cách Giao Diện Dashboard Sử Dụng Lớp Này

Trang **Document Upload** (`01_document_upload.py`) sử dụng chính xác kiến trúc được kiểm tra phía trên:
1. Người dùng kéo thả file vào Streamlit uploader.
2. Streamlit lưu file vật lý vào thư mục cục bộ `data/raw/` của dự án.
3. Giao diện khởi tạo `RAGPipeline` (đã nạp các cấu hình từ Sidebar) và gọi:
   ```python
   result = pipeline.index_document(file_path)
   ```
4. Hiển thị thông báo màu xanh lá nếu `result.success` là `True` hoặc bảng lỗi màu đỏ chứa `result.error_message` nếu có sự cố xảy ra.

---
**Bài học tiếp theo:** Ở Sprint 2, chúng ta sẽ bắt đầu viết logic phân tách văn bản thật (`TextChunker`) để chia nhỏ tài liệu thay vì dùng ước lượng độ dài giả lập.